# Duplicated name requests


In [ ]:
import os
import sys
import logging
import pandas as pd
import json
import gspread
import sqlalchemy
from cloud_sql_connector import DBConfig, getconn
from oauth2client.service_account import ServiceAccountCredentials
from df2gspread import df2gspread as d2g
module_path = os.path.join(os.getcwd(), "nr_duplicates_report")
if module_path not in sys.path:
    sys.path.insert(0, module_path)
from config import Config
from notebookreport import report_run_date


In [ ]:
config = DBConfig(
    instance_name=os.getenv('DATABASE_INSTANCE_CONNECTION_NAME'),
    database=os.getenv('DATABASE_NAME'),
    user=os.getenv('DATABASE_USERNAME'),
    ip_type='public',
    schema=os.getenv('DATABASE_SCHEMA'),
)

def get_conn():
    """Create Cloud SQL connection using Cloud SQL connector."""
    return getconn(config)

engine = sqlalchemy.create_engine(
    'postgresql+pg8000://',
    creator=get_conn
)


In [ ]:
report_date = report_run_date()
query = """
WITH candidates AS (
    SELECT
        r.id,
        r.nr_num,
        n.name,
        r.submitted_date,
        r.state_cd,
        btrim(CASE
            WHEN coalesce(btrim(n.designation), '') = '' THEN n.name
            WHEN right(upper(btrim(n.name)), char_length(btrim(n.designation))) = upper(btrim(n.designation))
            THEN left(btrim(n.name), char_length(btrim(n.name)) - char_length(btrim(n.designation)))
            ELSE n.name
        END) AS name_without_designation
    FROM requests r
    JOIN names n ON r.id = n.nr_id
    WHERE r.state_cd = 'DRAFT'
      AND r.nr_num NOT LIKE 'NR L%'
      AND n.choice = 1
      AND (r.submitted_date AT TIME ZONE 'America/Vancouver')::date
          >= (CURRENT_TIMESTAMP AT TIME ZONE 'America/Vancouver')::date - INTERVAL '7 days'
)
SELECT
    id AS "ID",
    nr_num AS "NR NUMBER",
    name AS "NR NAME",
    to_char(submitted_date AT TIME ZONE 'America/Vancouver', 'YYYY-MM-DD')
        AS "REQUESTED ON DATE",
    state_cd AS "STATE",
    '' AS "NOTES"
FROM candidates
WHERE name_without_designation IN (
    SELECT name_without_designation
    FROM candidates
    GROUP BY name_without_designation
    HAVING COUNT(*) > 1
)
ORDER BY name, nr_num;
"""

with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text(query))
    name_requests = pd.DataFrame(result.fetchall(), columns=result.keys())

result_filename = os.path.join(os.getcwd(), 'nr_duplicates_report/data/', 'nr_duplicates_' + report_date + '.csv')
with open(result_filename, 'w') as f:
    if name_requests.empty:
        f.write('none')
    else:
        name_requests.to_csv(f, sep=',', encoding='utf-8', index=False)


Upload the data to google storage.


In [ ]:
if Config.ENVIRONMENT == 'prod':
    if not name_requests.empty:
        dictionary = {
            "type": Config.TYPE,
            "project_id": Config.PROJECT_ID,
            "private_key_id": Config.PRIVATE_KEY_ID,
            "private_key": Config.PRIVATE_KEY,
            "client_email": Config.CLIENT_EMAIL,
            "client_id": Config.CLIENT_ID,
            "auth_uri": Config.AUTH_URI,
            "token_uri": Config.TOKEN_URI,
            "auth_provider_x509_cert_url": Config.AUTH_PROVIDER_X509_cert_URL,
            "client_x509_cert_url": Config.CLIENT_X509_CERT_URL
        }

        file_path = os.path.join(os.getcwd(), "nr_duplicates_report/data/") + "service_key.json"
        with open(file_path, "w") as outfile:
            json.dump(dictionary, outfile)

        scope = ['https://www.googleapis.com/auth/spreadsheets']
        credentials = ServiceAccountCredentials.from_json_keyfile_name(file_path, scope)
        gc = gspread.authorize(credentials)

        sheet = d2g.upload(name_requests, '1KFo3oUyzXo9A1aAOSy8cjR5ArxVT2Uvgdbe8NEZNLJU', report_date, credentials=credentials, col_names=True, row_names=False)
else:
    logging.info('Skipping upload to Google Storage')
